In [ ]:
# !pip install tokenizers

In [30]:
import re
from tqdm import tqdm
import unicodedata
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.pre_tokenizers import Whitespace
from tokenizers import normalizers
from tokenizers.normalizers import NFD, StripAccents
from tokenizers import pre_tokenizers
from tokenizers.pre_tokenizers import Digits
from tokenizers.processors import TemplateProcessing
from tokenizers import decoders
from tokenizers import trainers
import os
from os.path import join
import time

In [11]:
import nltk
from nltk.corpus import stopwords

In [14]:
# print(stopwords.words('english')) #'arabic',

In [15]:
def english_preprocess(s: str):
    '''Remove non english characters and unnecessary spaces.
    @input: string
    @return: cleaned string
    '''
    
    s = re.sub(r'[" "]+', " ", s)
    s = re.sub(r"[^\sa-zA-Z0-9?:()!,،'.:]+", "", s)
    
    s = s.rstrip().strip()
    return s

def gen(files: list, preprocess):
    '''Line generator.
    take list of files and iterate line by line on each file.
    @input: files       --> list of file pathes
            preprocess  --> addres of the function that will be applied on each line.
    
    @return: line by line from each file 
    '''
    i = 0
    for ff in tqdm(files):
        with open(ff, 'r', errors='ignore') as f:
            for line in f:
                yield preprocess(line)

In [16]:
def get_tokenizer(files: list, vocab_size: int, save_dir: str, file_name: str, preprocess):
    '''Train and save Whitespace tokenizer.
    @inputs: file        --> list of file pathes.
             vocab_size  --> size of generated dictionary.
             save_dir    --> save dir.
             file_name   --> name of saved tokenizer
             preprocess  --> addres of the function that will be applied on each line.
             
    @return: trained tokenizer    (you can dismiss that)
    '''

    unk_token = "<UNK>"  # token for unknown words
    spl_tokens = ["<UNK>", "<SEP>", "<MASK>", "<CLS>", "<PAD>"]  # special tokens
    tokenizer = Tokenizer(WordPiece(unk_token = unk_token))
    

    normalizer = normalizers.Sequence([NFD(), StripAccents()])
    pre_tokenizer = Whitespace()
    pre_tokenizer = pre_tokenizers.Sequence([Whitespace(), Digits(individual_digits=False)])
    
    tokenizer.normalizer = normalizer
    tokenizer.pre_tokenizer = pre_tokenizer
    tokenizer.post_processor = TemplateProcessing(single="<CLS> $A <SEP>",
                                                  pair="<CLS> $A <SEP> $B:1 <SEP>:1",
                                                  special_tokens=[("<CLS>", 1), ("<SEP>", 2)])

    tokenizer.decoder = decoders.WordPiece()
    trainer = trainers.WordPieceTrainer(vocab_size=vocab_size, special_tokens=spl_tokens)
    try:
        print('Start training')
        tokenizer.train_from_iterator(gen(files, preprocess), trainer)
        print('finish training')
    except KeyboardInterrupt:
        print('trainign stop')
        
    tokenizer.save(f"{save_dir}/{file_name}.json")
    
    return tokenizer.from_file(f"{save_dir}/{file_name}.json")

In [17]:
from tokenizers import Tokenizer
import re
import yaml
from yaml import CLoader
from pyarabic.trans import normalize_digits
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
from os.path import join
import pandas as pd

In [18]:
import re
from tqdm import tqdm
import unicodedata
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.pre_tokenizers import Whitespace
from tokenizers import normalizers
from tokenizers.normalizers import NFD, StripAccents
from tokenizers import pre_tokenizers
from tokenizers.pre_tokenizers import Digits
from tokenizers.processors import TemplateProcessing
from tokenizers import decoders
from tokenizers import trainers

In [19]:
class TokenHandler:
    def __init__(self, json_path: str, lang='en', ):
        self.tok = Tokenizer.from_file(json_path)
        self.tok.enable_padding(pad_id=self.get_id("<PAD>"), pad_token="<PAD>")
        if lang == 'en':
            self.pre = self.english_preprocess
        elif lang == 'ar':
            self.pre = self.arabic_preprocess
        else:
            raise NotImplementedError('This class suports En and Ar language only for now')

    
    def arabic_preprocess(self, s: str):
        '''Remove non arabic characters and unnecessary spaces.
        @input: string
        @return: cleaned string
        '''
        s = normalize_digits(s, source='all', out='west')
        s = re.sub(r'[?]+', "؟", s)
        s = re.sub('[" "]+', ' ', s)
        s = re.sub('[^\sء-ي؟:()!,،.:1-9]+', '', s)
        s = s.rstrip().strip()
        return s
    
    def english_preprocess(self, s: str):
        '''Remove non english characters and unnecessary spaces.
        @input: string
        @return: cleaned string
        '''

        s = re.sub(r'[" "]+', " ", s)
        s = re.sub(r"[^\sa-zA-Z0-9?:()!,،'.:]+", "", s)

        s = s.rstrip().strip()
        return s
        
    
    def __call__(self, text, length=None):
        text = self.pre(text)
        out = self.tok.encode(text)
        if length is not None:
            out.pad(length, pad_id=self.get_id("<PAD>"), pad_token="<PAD>")
            out.truncate(length)            
        return out.ids
        
    def encode(self, text: str):
        '''@input: text --> single string.
        @return:  ids, tokens
        '''
        text = self.pre(text)
        out = self.tok.encode(text)
        return out
    
    def get_id(self, token: int):
        '''@input: token --> single word 
        @return: id --> int
        '''
        return self.tok.token_to_id(token)
    
    def encode_batch(self, data: list):
        '''@input: data --> list of strings.
        @return:  ids, tokens
        '''
        data = tuple(map(self.pre, data))
        output = self.tok.encode_batch(data)
        return output
    
    def decode(self, ids: list):
        '''@input: ids --> list of int
        @return: text --> single string.
        '''
        return self.tok.decode(ids)
    
    def decode_batch(self, ids: list):
        return self.tok.decode_batch(ids)

In [7]:
class TokenHandler:
    def __init__(self, json_path: str, lang):
        self.tok = Tokenizer.from_file(json_path)
        self.tok.enable_padding(pad_id=self.get_id("<PAD>"), pad_token="<PAD>")
        if lang == 'en':
            self.pre = self.english_preprocess
        elif lang == 'ar':
            self.pre = self.arabic_preprocess
        else:
            raise NotImplementedError('This class suports En and Ar language only for now')
    def arabic_preprocess(self, s: str):
        '''Remove non arabic characters and unnecessary spaces.
        @input: string
        @return: cleaned string
        '''
        s = normalize_digits(s, source='all', out='west')
        s = re.sub(r'[?]+', "؟", s)
        s = re.sub('[" "]+', ' ', s)
        s = re.sub('[^\sء-ي؟:()!,،.:1-9]+', '', s)
        s = s.rstrip().strip()
        return s
    def english_preprocess(self, s: str):
        '''Remove non english characters and unnecessary spaces.
        @input: string
        @return: cleaned string
        '''

        s = re.sub(r'[" "]+', " ", s)
        s = re.sub(r"[^\sa-zA-Z0-9?:()!,،'.:]+", "", s)

        s = s.rstrip().strip()
        return s
    def encode(self, text: str):
        '''@input: text --> single string.
        @return:  ids, tokens
        '''
        text = self.pre(text)
        out = self.tok.encode(text)
        return out.ids
    
    def get_id(self, token: int):
        '''@input: token --> single word 
        @return: id --> int
        '''
        return self.tok.token_to_id(token)
    

In [36]:
class MuSTCDataset:
    def __init__(self, root, tokenizers=None):
        self.root = root 
        self.tokenizer = tokenizers
        
        self.split = ['train', 'dev', 'tst-COMMON']
        files = dict()
        lang = ['en', 'ar']
        txt_dir = os.listdir(join(root, 'train', 'txt'))
        files['ar'] = join(root, 'train', 'txt', 'train_ar.txt')
        files['en'] = join(root, 'train', 'txt', 'train_en.txt')

        if self.tokenizer is None:
            self.tokenizer = dict()
            for l in lang:
                self.tokenizer[l] = self.get_tokenizer(files[l], 500, os.getcwd(), f'{l}', l)
        else:
            for l in lang:
                self.tokenizer[l] = TokenHandler(self.tokenizer[l], lang=l)
        
    def get_tokenizer(self, file, vocab_size: int, save_dir: str, file_name: str, lang: str):
        def english(s):

            s = re.sub(r'[" "]+', " ", s)
            s = re.sub(r"[^\sa-zA-Z0-9?:()!,،'.:]+", "", s)

            s = s.rstrip().strip()
            return s
        
        def arabic(s):
        
            s = normalize_digits(s, source='all', out='west')
            s = re.sub(r'[?]+', "؟", s)
            s = re.sub('[^\sء-ي؟:()!,،.:1-9]+', '', s )
            s = s.rstrip().strip()
            return s
        
        def gen(file: list, preprocess):
                with open(file, 'r', errors='ignore') as f:
                    for line in f:
                        yield preprocess(line)
                        
        preprocess = english if lang == 'en' else arabic
        
        unk_token = "<UNK>"  # token for unknown words
        spl_tokens = ["<UNK>", "<SEP>", "<MASK>", "<CLS>", "<PAD>"]  # special tokens
        tokenizer = Tokenizer(WordPiece(unk_token = unk_token))


        normalizer = normalizers.Sequence([NFD(), StripAccents()])
        pre_tokenizer = Whitespace()
        pre_tokenizer = pre_tokenizers.Sequence([Whitespace(), Digits(individual_digits=False)])

        tokenizer.normalizer = normalizer
        tokenizer.pre_tokenizer = pre_tokenizer
        tokenizer.post_processor = TemplateProcessing(single="<CLS> $A <SEP>",
                                                      pair="<CLS> $A <SEP> $B:1 <SEP>:1",
                                                      special_tokens=[("<CLS>", 1), ("<SEP>", 2)])

        tokenizer.decoder = decoders.WordPiece()
        trainer = trainers.WordPieceTrainer(vocab_size=vocab_size, special_tokens=spl_tokens)
        
        tokenizer.train_from_iterator(gen(file, preprocess), trainer)

        tokenizer.save(f"{save_dir}/{file_name}.json")

        return TokenHandler(f"{save_dir}/{file_name}.json", lang)

        
    def setup(self):
        self.data = self._load_from_dir(self.root)
        
    def _get_yaml_data(self, path):
        with open(path) as f:
            data = yaml.load(f, Loader=CLoader)
            return data    
    
    def _get_text_data(self, path):
        with open(path, 'rt', encoding='utf-8', errors='ignors') as f:
            return f.readlines()
       
    def _load_from_dir(self, root):
        # split_name = root.split(os.sep)[-1]
        
        final_data = dict()
        for spl in self.split:
            txt_dir = os.listdir(join(root, spl, 'txt'))
            wav_dir = join(root, spl, 'wav')
            files   = dict()
            for f in txt_dir:
                if 'yaml' in f:
                    files['yaml'] = join(root, spl, 'txt', f)
                elif 'ar' in f:
                    files['ar'] = join(root, spl, 'txt', f)
                elif 'en' in f:
                    files['en'] = join(root, spl, 'txt', f)
                else: None
                
            data = self._get_yaml_data(files['yaml'])
            data = pd.DataFrame.from_records(data)
            
            data['wav'] = data['wav']#.apply(lambda x: os.path.join(wav_dir, x))

            data['en'] = self._get_text_data(files['en'])
            
            data['ar'] = self._get_text_data(files['ar'])
            data.drop(columns=['offset'], axis=1, inplace=True)
            final_data[spl] = data
        return final_data
    
    def count_words(self):
        start = time.time()
        sw_en = set(stopwords.words('english'))
        sw_ar = set(map(lambda x: re.sub('[^\sء-ي]+', '', x), stopwords.words('arabic')))
        def fun_en(word):
            word = re.sub(r"[^\sa-zA-Z]+", "", word)
            if word not in sw_en:
                return word
            return ''
        
        def fun_ar(word):
            word = re.sub('[^\sء-ي]+', '', word)
            if word not in sw_ar:
                return word
            return ''
        
        for spl in tqdm(self.split):
            
            self.data[spl]['en_words'] = self.data[spl]['en'].apply(lambda x: len(x.split()))
            self.data[spl]['ar_words'] = self.data[spl]['ar'].apply(lambda x: len(x.split()))
            
            self.data[spl]['word_ratio'] = self.data[spl]['ar_words'] / self.data[spl]['en_words']
            self.data[spl]['wav-word ratio'] = self.data[spl]['duration'] / self.data[spl]['en_words']
            
            print('count words done      .', end='\r')
            
            self.data[spl]['ar_tokens'] = self.data[spl]['ar'].apply(lambda x: len(self.tokenizer['ar'].encode(x)))
            self.data[spl]['en_tokens'] = self.data[spl]['en'].apply(lambda x: len(self.tokenizer['en'].encode(x)))
            print('count tokens done     .', end='\r')

            
            
            self.data[spl]['en'] = self.data[spl]['en'].apply(str.lower)
            self.data[spl]['en'] = self.data[spl]['en'].apply(lambda x: ' '.join(map(fun_en, x.split())))
            self.data[spl]['ar'] = self.data[spl]['ar'].apply(lambda x: ' '.join(map(fun_ar, x.split()))) 
            print('stop word removal done.', end='\r')

        end = time.time()
        print('Calculation take around:', end - start)


    
    def get_train(self):
        return self.data[self.split[0]]
    
    def get_dev(self):
        return self.data[self.split[1]]
    
    def get_test(self):
        return self.data[self.split[2]]
    
    

In [37]:
dir_path = '/kaggle/input/must-c-en-ar'
token = {'ar': '/kaggle/input/helper-for-s2t/tokenizers/ar_tokenizer.json',
         'en': '/kaggle/input/helper-for-s2t/tokenizers/en_tokenizer.json'
        }
my_data = MuSTCDataset(root=dir_path, tokenizers=token)
my_data.setup()

In [38]:
my_data.count_words()

  0%|          | 0/3 [00:00<?, ?it/s]

 33%|███▎      | 1/3 [01:12<02:24, 72.25s/it]

 67%|██████▋   | 2/3 [01:12<00:29, 29.98s/it]

100%|██████████| 3/3 [01:13<00:00, 24.44s/it]

Calculation take around: 73.33191871643066


In [39]:
train = my_data.get_train()
dev = my_data.get_dev()
test = my_data.get_test()

In [40]:
train.head()

,duration,speaker_id,wav,en,ar,en_words,ar_words,word_ratio,wav-word ratio,ar_tokens,en_tokens
0,17.07,spk.1,ted_1.wav,truly great honor opportunity come st...,شكرا جزيلا كريس أنه فعلا شرف عظيم أصعد المنص...,19,17,0.894737,0.898421,45,45
1,11.33,spk.1,ted_1.wav,blown away conference want thank ...,لقد بهرت فعلا بهذا المؤتمر وأريد أشكركم جميعا...,30,17,0.566667,0.377667,43,48
2,13.78,spk.1,ted_1.wav,say sincerely partly mock sob need laughter,وأقول بكل حد تنهد حزين لأنني أحتاج ضحك ...,13,18,1.384615,1.060000,46,36
3,2.38,spk.1,ted_1.wav,flew air force two eight years,لقد طرت القوات الجوية لمدة سنوات,9,8,0.888889,0.264444,19,22
4,11.85,spk.1,ted_1.wav,take shoes boots get airplane laught...,والآن أجد نفسي مضطرا لخلع حذائي صعود الطائرة ...,17,11,0.647059,0.697059,39,43


In [41]:
train.to_csv('train.csv', index=False)
dev.to_csv('dev.csv', index=False)
test.to_csv('test.csv', index=False)

In [ ]:
from wordcloud import WordCloud
from textwrap import wrap

# Function for generating word clouds
def generate_wordcloud(data,title):
    wc = WordCloud(width=400, height=330, max_words=150,colormap="Dark2").generate_from_frequencies(data)
    plt.figure(figsize=(10,8))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis("off")
    plt.title('\n'.join(wrap(title,60)),fontsize=13)
    plt.show()

# Transposing document term matrix
df_dtm=df_dtm.transpose()

# Plotting word cloud for each product
for index,product in enumerate(df_dtm.columns):
    generate_wordcloud(df_dtm[product].sort_values(ascending=False),product)